# Optimal Synthesis Construction

This notebook demonstrates the construction of optimal synthesis for production-regeneration systems in membrane filtration.

In [ ]:
using Pkg
Pkg.activate("..")
using CTMembraneFiltration
using Plots
using LinearAlgebra

## System Parameters

Define the production-regeneration system parameters:

In [ ]:
# System parameters
params = (
    lp = x -> x^2,           # production cost
    lr = x -> 0.5*x,         # regeneration cost  
    fp = x -> -0.1*x,        # production dynamics
    fr = x -> 0.2*(1-x),     # regeneration dynamics
    gp = x -> 1.0,           # production
    gr = x -> 0.0            # no production during regeneration
)

println("System parameters defined")

## Hamiltonian Function

The Hamiltonian for the production-regeneration system:

In [ ]:
function hamiltonian(x, p, q, u)
    lp, lr, fp, fr, gp, gr = params
    
    # Production and regeneration contributions
    H_prod = (1 + u) / 2 * (lp(x) + p * fp(x) + q * gp(x))
    H_regen = (1 - u) / 2 * (lr(x) + p * fr(x) + q * gr(x))
    
    return H_prod + H_regen
end

# Test the Hamiltonian
x_test = 1.0
p_test = 0.5
q_test = 1.0
u_test = 1.0

H_test = hamiltonian(x_test, p_test, q_test, u_test)
println("Hamiltonian at test point: $H_test")

## Optimal Control Law

Compute the optimal control based on the Hamiltonian comparison:

In [ ]:
function optimal_control(x, p, q)
    H_plus = hamiltonian(x, p, q, 1.0)
    H_minus = hamiltonian(x, p, q, -1.0)
    
    if H_plus < H_minus
        return 1.0  # Production
    elseif H_plus > H_minus
        return -1.0 # Regeneration
    else
        return 0.0  # Singular arc
    end
end

# Test optimal control
u_opt = optimal_control(x_test, p_test, q_test)
println("Optimal control at test point: $u_opt")

## Switching Surface

Compute the switching surface where H(x,p,+1) = H(x,p,-1):

In [ ]:
function switching_condition(x, p, q)
    H_plus = hamiltonian(x, p, q, 1.0)
    H_minus = hamiltonian(x, p, q, -1.0)
    return H_plus - H_minus
end

# Find switching surface for given q
q_fixed = 1.0
x_range = 0.0:0.1:3.0
p_range = -2.0:0.1:2.0

switching_points = []
for x in x_range
    for p in p_range
        if abs(switching_condition(x, p, q_fixed)) < 0.01
            push!(switching_points, (x, p))
        end
    end
end

println("Found $(length(switching_points)) switching points")

## Visualize Control Regions

In [ ]:
# Create control region plot
x_grid = 0.0:0.05:3.0
p_grid = -2.0:0.05:2.0

control_map = zeros(length(p_grid), length(x_grid))

for (i, p) in enumerate(p_grid)
    for (j, x) in enumerate(x_grid)
        control_map[i, j] = optimal_control(x, p, q_fixed)
    end
end

p1 = heatmap(x_grid, p_grid, control_map, 
    title="Control Regions in Phase Space",
    xlabel="State x", ylabel="Adjoint p",
    color=:RdBu, colorbar_title="Control u",
    clims=(-1, 1)
)

# Add switching points
if !isempty(switching_points)
    switch_x = [pt[1] for pt in switching_points]
    switch_p = [pt[2] for pt in switching_points]
    scatter!(p1, switch_x, switch_p, 
        markersize=2, markercolor=:black, label="Switching surface"
    )
end

display(p1)

## Optimal Trajectory Computation

Compute optimal trajectories for different initial conditions:

In [ ]:
function simulate_optimal_trajectory(x0, p0, q0, T, dt=0.01)
    n_steps = Int(T / dt)
    
    # Initialize arrays
    t = range(0, T, length=n_steps)
    x = zeros(n_steps)
    p = zeros(n_steps)
    u = zeros(n_steps)
    
    # Initial conditions
    x[1] = x0
    p[1] = p0
    u[1] = optimal_control(x0, p0, q0)
    
    # Simulate
    for i in 2:n_steps
        # Current control
        u[i] = optimal_control(x[i-1], p[i-1], q0)
        
        # Dynamics (simplified)
        if u[i-1] == 1.0  # Production
            x[i] = x[i-1] + dt * params.fp(x[i-1])
        elseif u[i-1] == -1.0  # Regeneration
            x[i] = x[i-1] + dt * params.fr(x[i-1])
        else  # Singular
            x[i] = x[i-1]  # Constant state
        end
        
        # Simple adjoint dynamics (placeholder)
        p[i] = p[i-1]  # Would need proper adjoint equations
    end
    
    return t, x, p, u
end

# Test trajectory computation
x0_test = 1.5
p0_test = -0.5
q0_test = 1.0
T_test = 10.0

t_traj, x_traj, p_traj, u_traj = simulate_optimal_trajectory(x0_test, p0_test, q0_test, T_test)

println("Trajectory computed with $(length(t_traj)) time steps")

## Visualize Optimal Trajectory

In [ ]:
# Create trajectory plots
p2 = plot(t_traj, x_traj, 
    title="Optimal State Trajectory",
    xlabel="Time", ylabel="State x",
    label="State", linewidth=2
)

p3 = plot(t_traj, u_traj, 
    title="Optimal Control",
    xlabel="Time", ylabel="Control u",
    label="Control", linewidth=2
)

# Phase portrait
p4 = plot(x_traj, p_traj, 
    title="Phase Portrait",
    xlabel="State x", ylabel="Adjoint p",
    label="Trajectory", linewidth=2
)

plot(p2, p3, p4, layout=(1,3), size=(1200, 400))
display(current())

## Multiple Initial Conditions

Compare optimal trajectories for different initial conditions:

In [ ]:
# Different initial conditions
initial_conditions = [
    (x0=0.5, p0=-0.5),
    (x0=1.0, p0=-0.5), 
    (x0=1.5, p0=-0.5),
    (x0=2.0, p0=-0.5)
]

colors = [:blue, :red, :green, :orange]

p5 = plot(title="Optimal Trajectories for Different Initial Conditions",
    xlabel="State x", ylabel="Adjoint p", legend=:topright
)

for (i, ic) in enumerate(initial_conditions)
    t_i, x_i, p_i, u_i = simulate_optimal_trajectory(ic.x0, ic.p0, q0_test, T_test)
    plot!(p5, x_i, p_i, 
        label="x0=$(ic.x0)", 
        color=colors[i], linewidth=2
    )
end

display(p5)

## Summary

This notebook demonstrated:
1. Hamiltonian formulation for production-regeneration systems
2. Optimal control law computation
3. Switching surface analysis
4. Control region visualization
5. Optimal trajectory simulation

The optimal synthesis provides a complete characterization of all optimal solutions depending on initial conditions.